# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': 'e6c7ff3f7c47b499702730b8407fe7947c13fd460942d044e326740418de8a7e',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_olmgswZ1yWfvloFQe;!E6eD_M>&9iMjWESHl^uW}uklBg0}6shK=%5sg?nm;%%^Eu~h<_FF%oP7a600dQK$2~pM)9#K<'
            'k-)}nV`Ja3zrX)FuFGhW#^Twt)jGZqMV!sz;$WSu<21=)v53=kT$G|*L`6IklZ`mKKjuPXRcMf`H`zq=t`^C3v6p1{t`t!w;=k0%'
            'Wt7HQ6^h4s8chxosJO=3StX|VdL#0=s1`{n()=Qs_My^DT*gH*-;C3I8l{8vW{=^DWVOzVs^sFUd{#rJXGsxHt8^pgc>y0{L6zhn'
            'M7heBv8dv*>JRqzUgl6UyMPfDv8dN+9?ePt)z<YSP09uJctb@7;&~<V48RsK7Gj|DB&9MlaaAM$bkBTUQ-hI!Hz`bCS+Cb=0`ryE'
            ')w-_4I;s|fz5V_Dy?K$Z#CSZftGbBCV>v1So8?thC3#lv?a6nGs9dDUME(48nP=*`h}BcMsH-H^zv{`l$ft2xsyEdF#{h%kmS<5F'
            'SIH_??W*6Qz`sA_S<L0I!_c-W_7nWv+dF;p=*`ck<6j=V`1$Fn7>VmW0e|-6_kbZ#JFB8}KNS13_%crOHNF_H(`Xtm@)TwVKNaz1'
            '5?=xS0Ac_-?S~8@T0^fd@weDdva+7flWBt0p?eX&oJTOr^cf1y>M2gt7zefi5L@Wk&)>g!{^rHg@vlETKmF<H>+z4To;-a)eGIB3'
            '+XSHq$`z~)KCI$dQm^nSjf#u-;PTD^EXE`c0MO${KfQVW>gD*&4^Mx5I{xYPt7p$&7-J1)@jR;2Dj3Qa^b-1nSTF{}EumxZ@i=V0'
            '&T|+@7Oi4<A3TZw9Q{&@(<mzgw@jIQz%t*Q7#|`S>Sa8h=4oDxqq0O8|NO%d{CNg{Oc0Z(^ptz{%q_Qy-jA>HVunTTI`5~j$?^T2'
            'BkMzPk!0gbcz<$YegGVgizze<U!Fy285`fyxT=6cl<Q~;xCaGf#V!96T)=#c^Z6Li6L7A)X+BT7=JUs>n8s;Zm*Vts=Y$^p%_sEa'
            '!N*SMo1)%5p92$QRRBUwi)2k31LkkFMjY+w5>)C*6tD73yn*(kViRYfc)f_?Y*R$$yI+zrE;gn4>c>qP6-fj;#s>I9UYD_1@K>O<'
            'agmpycv3_cK+Gyw-BA1y`usL1#S5UG@YsU$yufY|DnR>Dx`D>9Coyc$4IqFn-FKH5@uY<dr;iU_#}_qJ9#qx5jTVdO<lcR(adhYC'
            '?%il||K7>TgUQjodv~Y5i|$R2<NJ4J^ON7r?#A~XMBf}gxc5zba(wdO=s1ez-`syNo6nAq?oVd(qZa78ND=O$s@CQ3@UV!k1{X=S'
            's3$N_#S}^aQ5#J2)!{{+U!?INRw@ryQIZ|z^AsNB2+HZ<Ek=_j(>MbpH7DWKvloNvedR0m89V1`WdcVa&}g8_X+&ELj}UARHct$2'
            '6f_Cs;C4e)P}LYd0=nHtNqD{#e~i~~+%kLe9uDbgf}F@|4G0bg>|v3o=_HyiMNWHZARg6KzJfNV;t#Lhhyo6DWFoTsO2UK5D~a`3'
            '<-k|K0bC_j8Xw@{M0}M=p@gJtBAUQ)H}I0eF>LS;ar((8%ww+JK@0Cc_}kOMZ$39IJRY3vq6<7D2frW*gB7`fCd}{79^9GTxjQ?a'
            '-=BVS_xSjCcjCLzbbfaNL}C8Bdv}iGd(r)GCikO==*H~mDEekL`Q{|L8z0T@-2L1%fzjcR2k}q$zQ4LyRR47E$={Y1$WDDS8L$S`'
            'xiTNO`MrJG5_|ZEN5IUzdIntG>sK$Io=WcSk5B*f>#NsKkh|+OTahIa=1Sn8TF?<k??Bw3Lk(Y7>deC*SMt=8Z3YP*XAA>B&zhr<'
            'e$pw7PlY^v2}S~QUd&@;QTO_LdwalX0-ImvDelmS4|eFoD()foj4jUi_fV8o0b>Xvpi}^gR_}c^#=rEppf3)-1#xLw4VmRe$&Y}8'
            '&@zjm=~a>?C6YPWn5I=E4`@<JO`_POkOYMqgs6K3%|WAXFR=>uus?v$$-37cr1@1`^!ijinHvzkP@@@YWU#)2tR#|5_?h*g#ak2+'
            'urtV0)bZ1z$P1e#!awu<?-P(%S23)n9=x2{Yu)A-#rC1t_f%fRms6DRXtqH1gGwccO!2Je?_PgvL+R^Za<yk6ik)y4!9?oj0YlUh'
            '@O&~)6a&}Yj5o^*ka2*}&&Co7<XV_0K?@PslaGJkjGHcK^)e`BPg8FhZ;-5$n5~{+0dCk=ux3W#cJB;s4*$GF_HVB}F91e%&RN>_'
            'e6{3?P|O3_hvBt^vTgWDd;YqBDxnV+A<KFC#}eq6U<PsrfEbyX$F{QvfMhibYL?<AjY6^yPMKPXy(O^#{1CRd1o(s-y#xQw6!ddB'
            '{aZ<F<QYgO$ar3-X%ERP(G9>Mo~2oG0lkMgk-CWi*O6#+1t$pbO>sswk-$xW?f|c~h>(ys6OCvzUDVkU@gf1@(@Ue(WEKr&$pLat'
            'z5Dm>-MQZn#RSmRUL@|upkAZ;qDPSW<`T&7i}*c1^%OycV+7SM7Cjx~@zLW)m>XDVo<E?Z5unR>mK1H``D3(1%Bg82fDqATl%&M-'
            'k!C2}7}_itWhvH4mcb4?{o&C8Oxj?NfPK%0oZzzz<`8(!=b0#hwE;Ggpp)thRbDZjLh*ZG+~7z;B}tTuKhOzHu%6*TNE#`Tlwy`&'
            'WvJysHB>f*S6P)4fF}hUq6f(3!Y8RKqp<?F%am^>XOljm@&%P(^$ay)u(C1g+9K>Ds`lVz2`fJ(<r=gA=Sec3#|5hRa9###&gfq?'
            'V<TbC_?gDtb1J2{7$_z`3@+lT=kRv=<iPVly&W3#l=maok~{+u8RpDNT-xSUnb*LnBU@2!V27><FHk3hurw+_k5C!bYjDH<K3FZ`'
            'Z`c4h{&GaDE36nCt>b)2Ke@sT&@#zL;|Cxc@P~RJOyJ+I<0%pl-e}EyoLPM9xqFrx;0H-bQ`TecYI~a-W%aQjMkBZF8zvUd855E-'
            '$#e|qIkTplc1VK90)AIB4&(eYG(Rr#yi$`s=3bBc14+byiQt)*|4>cEHe$zk!Wuw_&HVW})pIq2CX^ee&@6nrIb^-<0CnIeR`t+a'
            'yXFvdc;U`*@jltjIfuvqn8&MAycw=dcD;R;bwUe=t<JPWldus@`b|F=A@ICapfzF;ZS-+GQaugQ3?4@EaNX0NVHd-9;M3hL77VUc'
            'Z?*$PCmXh)DmM1W>14B88Ng3I>#Pllg4H=|)APC--MvQ#AfW3SL;#1^u(~)PtHOA)sp7Jy%JV^JOp;u%HU?@q4f=ht1O06F_;sCO'
            'hnYbRk_=`EXNQk9d0fG9a#7T9PDf=TCJ`zG%j82~cdZ#s5tHh_=S61k9N&7LO&Qjh>XV}zrkXg}PsGyOuTK1aiW;V;^bb;Z=#zpC'
            '-`d?;7jQ85=E2W@di47Fkmo#2$|x(v5_TF+w^|XjT!4IGfZI9?&}<^bAB+`x>+NK)zn;Ff`UvROEGwO4rArv>rLWNdxa{@!Za_4T'
            'F!d&0MKl97aDpF_B8^}vqFEw(*Y-A}6*}McVK#6U1|HFD|5;Ql(}ci9qKr#GS8)+ffiv1f>m)_E14kB9!tQi$uXCq)kIP+q%`^0Q'
            'V2l){47JIY@TLGxA1$QVS+(RwqiV@N?<vd=TSxIQpQBNDJOOY;66<v*?E$u4ie6I$WUV=@(j>+o{ek%l1h|cu#KdZ8hAoFtivRex'
            'f9I)QMHg|B%NE)CS}u|`DQe|Q>c%GI+6+8|jtfLwXaPCUm`KPTOT&U0IaNK$$_m(Wt6zW&(|om}=CE@E@t7>QrC8M<=+tGbm=UOe'
            'YZ1X>OM~<rN|u6@#lZFg1A;uecqdysNZ$1=ZW#`-dWD8qLUp?jY=hUj6nU)KbFwEZs_91Ao)2W|4lu{`2_!C%uI#Ev;nZ5j?_;?&'
            '%|5hY(Kj2_I+j|=Rwam4%wYiQT;PsD$*wI#!YH3oSy(l)p7SaZyM~fsyqd%_v=iZqB_Q-P7{D+uifFw+cxpC#K*I-G_g;>EkFdT4'
            '6nva#K=mqKr814k%nyqN&0IyrOsH4L)p7qz0vL&3f#&8{AViU@#42Zj8t_c?9uILmkKeu>_Vtn(+7@O4)d3j1eTzrQVI!@;`O+ix'
            'hxk?;Q`7kC?D!lR#$aeDy(Ccjd~`0p765D*z`yk7WO#0_4!)C1F{zW3=zW!dAUZZ?t%p)1(SZ<l%B)%pQ7jC_B2F$AmHaVD>tb9c'
            '7psU~Ko9V2b(T;O0RQ65i|7I!07MVAVagr|hZRu56xVf9pzI0D!kRV$nL~GRcTN`(P!HTP=;i=$R<J&2N5ji=7GNWx#wC7z|ATU{'
            '00B8hDf)~m%+6Wp<iQ4TTB5;VTIB11)L%Hj0w^_N#YKU1D6pkeR5_uzjSFJdZplFQAvWKqjsha3@#vc7cRRf1x!n!{>89~GFOmzO'
            'V<|ATYhvG#P>+&qit3y=+;;$Cvn^0+l>3gh%<Q&jjJNFlI!%C^I-6XK%VK(dHs^mIx;;|`{_3L*FK{d)Ldel|gPdFFfu~z@U$+iU'
            'c6MkMbBCa^cc3^6=ZK+cv>Ek&J-F&0o(y|G94z49uEl6+p6C0MM$s&FD~zGn_|F%`R`e$E639cG4TXVryd&+!?C<xj%^uY=B=^Qn'
            'Uqq!bTKqf|lRQsBz^tcB*oI@^bK%sP$=6<fxly@69iA*;QGuOAGJ!_HOJG(|Q!d}Z0ZoN6`@^SFk5%#>=>_0rbS`>oG-8@2Ye7Te'
            'UQ1ssQaVk63cx;SkG{{+#n~~z(jDzHSviY<lAI3=0)SkGXAa#x1o(jvdAOr%cLKHySVMK^{uW!Z0dCAVB0oHNuyuED;ElhU{1B<_'
            'pCLapi_h@^mchF%u!1g#e-ZIUYdSEH<p_PxN_74j9Sx4|4SB0wH<+{?MpXqEJ%VpQy<u>pJ4Z)nSOE4@yFvLu^(cbBfRq6FFGnZC'
            'S<NFI!4Zu*-{A~)4ov-+d}9*&bOO+g<uLZDJNX|q@t-j1*7VW5|JK21n_-<vR?lF~<%u5^7v&Iz%jyh%gY5iOh^*}et(I+mQ5;BK'
            'p$I9vGcMz)+|lA+nVBEaKfJ%S^WoqzpaAep@$4sl(d_#gHoe^Ov>gvN;$Q)o1Eej>fFK^EsLj5NQxPN41YQR_JrJ*vnwFLn)z}<R'
            'sf{}Q8eLS&3J7Z<dKJvXG=e_#k<G~R?BGLO<fx0Sr0V_G1rXxzU%mN3yi>FAPV|5z<&!clfZ8+k!4wE?1#&*|YVX=p@(%jQOw0u^'
            'i^LogCl370RVLmYG{9lj<7kB(KdKIx?S%a@i)RM_C(mZ(L75-SBVgc5iI{6LCW`1HLivS;y37!+<a|^|2I38}OaM9p!7-~*7Fxlc'
            'f;mA_8cLa{BF7gH5LYlYL$wqj7*#O4i~I`Y9uRFL?1U&F`MXv)F5*?812hFpc_sT5XBRM2&^Y&P63@{ahSw!c)qKNjLIpT(P|b-S'
            'HAR=^4xkq7EXzU>j2qSvgr^HM$bqB>h*50RNMbm)mINQFS1~lYh%V!j?IUP?DP=!eyq#JQY=y8G5DllMy|nnn^Y-^6anz9R=?cxJ'
            'XQp@*9FWhciQ4pAVK@FD1c!&O5W~;+8hubcMM<LBO@PQfXd_fOKIU1M#s%d24Z}h+#OZ*^8sZl!D2|3VX=LVslC+h64@4{2FXGw('
            'M~WAE$Gh$900{Td@Z3~KNt%kKqz&aaO_SO8KyN85fNto+W2%?sh{>frskDC3GSRJGdo{VsYg97Y^Zpa*9ckuan5bhkKR*BIX()+J'
            '>-}zCO0O93Q1u)=vqr~#=}so87n<*2fikk)k?|0>6+3eeCFOJ|WkQ>ut?cPWf{Rz{YJ<&GC~Qmd3-Ae03CO?->cm-IUo1!{PGBeD'
            'uEYtYy*`81(j4_0Qb_w!P6UxJppPJQ<yQk~0b}|*hTa%w46gY3o-}#Q#h6BA>JM9bU=z?FT4S_8j}}Mbn$)^VKZZ_S<`8?E<$nPg'
            '_WKu4kB*M5M%GbT%3jel7i5r>uJVI7^wdKqpST)vpMh;A#~ss#$5o+UQBg^yiZvY6W>6GGk_kNom_BE~ix~JC>;bd}98u9z*o`K1'
            '+@<4WK86KDKf-th$W2GCtE81++L~GU!BNOAW5A1XC@xVzfwn1-LwV+`VL-%X^ny{)Y6+?;7o1)|TnSK+aZQ_1|6G-UM@x*JA4yin'
            'H=6=h35Y(W-nP_Ducp0`u)3RT-h8Q!JVgFd1TPOC1?F)zPYO6?Nu;x&eJS1<Lwko?jeE$)HqM0vJd4XU9!Cfb)zlB^ymNrT6qbPL'
            'MX{lBCEB{{EUBbEGsd1vXHR&<C$g6hY&QsIz*exKh^mEZ(hkM(80aznH#o8<O`(=#YDkvs&By|N+z$o+meBA8c4`!qS+J9)tpH+l'
            '5s&793?sTmki)^ze0z9J(CAr|nONqt6n0tbprBJ~B)7>`l&B3QSNVJ3z?;3@9F|m^-Ph0;q~A?f49Lo4bf-l=3_|{f3q5_m#!wX>'
            'tR)akXIHhluTNU(*N`D`cKgp%VK7Yros4>lzWh1I6_$Ozr&d&+6eEjZqM8tCf$@q=ra&qiH@gm|;G-A<hZpd(0bXP?kn1C{t&L?3'
            'YVAfd$&2ieQ23Z{Muj$Vb-XLlO6k0ri!XbI4qOUYTG{P*TC%bOO$2bDzmxR@vb$Fbn*24#^SgQ{{@3ZNmvnw`=omush(NfWjwxXt'
            'rR7G*a5!R)*?;Fh{_WpobXrNuRhFJ2^jp#%?r70a_6%P51C)Gc+PDE-tkW5C=C(<MaSP-Vlxp%hHAL<}7^|m(^jzhNIeoj3P#GhD'
            'i%Dy?2M!tp^YF?V*qPA;vWJ%@+Z|Zb-@M=zpx^l%|G~gJ!Ud_?KW~t8H$xt>a0e4H3hHWp@BpAt>*Xkr<^hT5{k>bU?HxN*&W_rm'
            'E^B^b?Mb;6a8Lt^f$XWmFbIX|JZqOpq^w2?Guk=@Kgxd(BIZFnpQ9Oza74&XZajuE!S<SWa=V3u_tKunVlY_)P`XoxEL(h_gN~)E'
            'PpmGQ{$gkcP6VN+hm40(67{rAJT+955^a=TMVpee1!bc=z)e(1F<aSt2_OZPBFl^t<*PRWk&49jlC~e<$UwY8Eu|LD;knLqlnD96'
            '^|G8D$e&2*WEe2}h=6fStUZw7wb;kB&p1)aacQ+Gtp*0|uAviYLe7f7+pS<N^o$URvE3!E1GOE7;_Ny=6r7H$&6@qRKuF8kSA9mt'
            '<|M2BRXWeHs@{0F4aHzEP+KLCn+i}fP@)%`qq*>wioHQ<fw90Sny(8q)yHf*g+6J!tz3KR#38q<C@bPa5nsgb`LIP>{-Rz*Srd)J'
            'b~nTkXr~D@)$zlNs63!?9^w;ed=K}uUobh~9;yK_V=eU*fY`vICg_kV!=r(XcJ&TS_4Ee1f#~tUt7+cDhCK5;%p!JVh*rraFK$l~'
            'P4U|myps+>_oa#tCvTOK7;BcBRv@!rm@b|fQa}?w{_z(X7d<C09&!>TZr1>)%(a4zwt~C_GT0&MdJ#?Hs(gr|ud>6wd#B0jyLTc-'
            'v-*|ds+>S94r>yRUY<S|L`2s>Aw-W%%kLvR_QiC4d~)wDEA()Kvp8!2V_A@-<XIB4hXL!TXxE~C8{?f3rO+oRL>uQNosi6xk=$*-'
            '!-iH7YW@Mx0<gJS0Ky=mi4?V1yzLP22WqDE!~&-N{N~xggNFt@d3=zO1Ti_9qyB3e)g=oav>^&MjAU^P0?MP)$IqWb9|{l!%fVmi'
            '1k*gJaR9NPE`@ESb|JTU3UC;daa2qfy`rIdZ_BTGZ)ackht1!;?}l#&`0KmA!ImLNnnW+WQzn;RLbW0MclN*EX6Ijh_Y0OE@(%|T'
            '+t6+g>Jb@)TA&(Jk2P4OYb2*|qtoVlvj_71?f7$-H00$b+9V=h)*B)o28(`tPq71X$(a2lJ&o+S(LK{ZReI;9_ucQmcsuLEDqWw1'
            '_qP4_`l<IFet?-V;0@4xx$YeqUUNjr0Vuk`=|*I^<9?rI7S%QhGdM50x;z5>^-pfDFDF2$nr=Mx=wI{5lV)no!x@V>ylpE~C)oC#'
            'ZNkx8y1Ol;YTyH7UX-W+$m%MF2c?%7IW>4*N>j?T8$;@kGf4pZ;wy3I{?U>6nr2r*$G7+K9jj(!Q+?Ta?U&ZD67nhR;seYsY_}E<'
            '$^u4<>)auFRW(rfGmhXu4W#Btf5IAYl3>Yqa9kiURy@TOnu2ryN7x)NwwD*5ux(GTf#$qm?&kTJTc-$%Y}QBUMXY`*(~kDBA_D2~'
            '*@4&M#q=DofQ(j3DXYd1Rp^6Rp?jJ+n7%gKXd;0uDe<xqAWT_0PHB==hJzT%68XVpXn2tX2*(nlwwb;FQ+aU_7Z!X|(DMp;E!~WE'
            'FzXbs<pPJzv~_EKTbf={`$eSPE|j*LI&8|gdUi;uy=R*+Zj^@1{@E)j(b?q=Z;8@j<5z0>Yw4G!<Cb4T2b<Pwzy-;2a1SU(^uFUW'
            'Zv4TKN1aiVgi*#AgaoWdmQ)+X3oA2aBpqCEUeU$cl}t$5ig+L33dOTPI;TV;U|0c{geK7a8OU&ro*(!Ldn5}3W(PB1l#;Y$gn<1w'
            'O>6QVWG_u69P{40ikM#?-mv}pQlKJ{gt1>oJF!g`#fK^K*GxND$i>YMJSXZ%kk`kPSjN^$IT1G<23+Lm>!|jv+Mdo)PP-H^l{it>'
            'O(SV)Y@obN4m(8?DWy`>DD<0hwD6RvDTe~|)|OOw{y7Q`Q{QZ*`G%&sc^yc6<*Ai{R9i_R=|rcX9oiRr2H>-RoT=HBJU|PP9b?lj'
            '3i3BG)SkjL9(skAza9n`Q?3K?tvK2<<t}gAh7FfKtB}HLJ3k4!0WlB5Q?3)<@Fza-oliCdBYtz57pM<2U>MskE0akY8{c@2mHgU<'
            'zU{ZevgGPEK;(uR%29^Ro*AlTl~Gp~Wkzh)rx-K!o)g)kT#^hr(|YC(1J@Ddi-rl|UYt2Nh2`Y)-mL-_cC<&6(JI(l+TBI1rS2~3'
            'S?=z_?d5Nmb>@e4nCFsRgBkd|SkTA>hyolR)96Xh&P~b7kRM7?qmU%8-ooZ9xxT^gQe>QDtj`U}>ytq-_k6y0<HDlrwbkuD(TN*Z'
            '8S7bn>|6B30@)zrRVa`OT3^qO&Up|lPV~c$&r-VRw=tl>E$Z4lxW3xHzg`$o?Kfay@5+iyWbq5h=)!u>J?ML)?rw$UmLOvaG4oYr'
            'A(qy+UpFEYl;MwBNZ92uV6?&#2~=;kRdC+2^7Y;T?hr>E8s_7ECsf^cy$B}9M1xi@_HN$U8ZfzQRE_lSa%$*iqzl8d%c{vhS=#3S'
            'H^M+vV2gT}ecC;jxO?QeM&|QU#j3Sw1V7xGNbG$n4nE%>fOe{sffsQW%Mtf(k#kb<POAu0u<U)Q{lWr}oPK^ee*N_2lc%r68g_LI'
            'Jl*H*HX8*AeaB_N&H=!Mtky~-_If`zk~yqr4>;_wB}W?qeGy~Qof)}@02jQ<XR2IC3_2G!#s0D>sVj#dud9-2nv_agLiX1lRXU(I'
            '_}EAJ{`lzV$i@~RG>u+hcyuz`4zEq6fboi;(C_e?8yp_p!Mq!$$NCfya*R59KxOjwE_>=h`VS$adCn3FC67|kD0fe2Mw>EPG-&{w'
            'q5dH%fUX3{NQ43EGy`ds5WP9!%K?Z8<r;bWY6F7uK%AoUvkU>sHbR0E<b^B4xO8vWZ+um!Rk9{6_9{lVG?Wbhdr?+_0cuege>6Ha'
            'I@~BEP3SJWQ9tnEjC*~q{BleOKk`?^6hC`A{cEgvBS}MeGVOZUSFM}knHzE!_{TX~PKnE>m&BZ59whAFkUFyM@9qiZ1(VTC1M)IA'
            '1_VII2EoOh(X<1&Y4-#KSChr8vt`Ktj8pRAJHs+|fiYE9(<&kD*>29ftlYYuG;~bZGX_=GD@;`&iym3NUMkfM4b^q>f=;Lz#_#?1'
            'XX^de%xzE+@wGU1C)I3!XIXLyQ&7L{TzR&XMQAf}pdXsk;_)g&*en>Ba|->i7V|5D-mdONIAf_azUhV<bPGhlPBEPN#lgJxnrjgL'
            'a=Va6&Ejpgi?8>--Qw@QB(n%w(n*dvhm)m;FKwcz)t!xvfbGH-jY($jLxhM(8E3Gz1trX>>8QYdj*bnkhXANyksf&V-q6!qpXdA6'
            '-jfetYgFiO;kf-)>?lYR)8FoQG(u&V1a_2A9-xh;K}PChtzPgplQT_~`fYQTa(GY)R3|aTuIf<ICQe8S1+0@(IvOX*nsHW**BHE2'
            '$IY2_gHy1OXVDmA!BWy;Xuv9UaT;Y8b#x&=SP>XEK~J)z!fXJX8`j7Rr_atedJ0P68#U|foP(2S&w*Z5J~`9mBEzr?I3DS!Z%n<k'
            'Z^WyNPFLv|A$n2=EmUgo>4ryxLPd?)Iw{gi#MN}rccQN89HESy){@ZKtjyv1L_RRoW3cf`5uvwaDLpmih=Y>#9J7Y?23C(k4jLsm'
            'Y#-?$QYgK@+*ER;V~(dc;)5-}VDwDjnH;N`lri7G^3NX;37SMEgcGn{=GR^uVK8^hWipLN!E{|yD!|pc#?1iY&M1I6c@S7fE-@Zs'
            'Ou!Y2G3hQx+|;1i4|YHrQUxD!z}BL4Xc`k#>Z8kOCZiHZb{bE4oXyD1cuZM~q3?NJ(gKd8(n4YUNIisN0&+5qc685%W~2D1Xa7bu'
            'FZoUTQp~BjK<a6*KLdxB53EzD91z=MXe!JZJ$IS4Hlx?+`eE;}NnSF2T4I7{^!f$NoYn16iyQfB4}Btwh1K2x?O!Y(+)n4_CNQyk'
            'zeqkrZq^vlqpThMhxYc=n(t+u!g?oZG>w)6W1JmIM>0rvB&_Jx8Z>Ckdd~4jDmuN}preJBhQe;Vtq5_oABerjmNWNmtCzQF1khr~'
            'dp6;e*&!u=jz7<W9;X&ZkCZGCn|#?fSZB%fpvrwD?pxr;lFY)UqSnrmVFK)7Q{$YO6x=%n26^?4Le0?PY2B3U8}U(yxRg9_<QwLp'
            'X5Wo2I9$Ax#jF`?wtWITt8PSZ{vLMyxbJ-H;OOkgRUBj3Bs^>wF|&ud%DKC(`D6*aTP5)GFSAV+t&*uqbXfLo!@S(UL8vsLpUmLg'
            'md&{pnBTw(-IgI@lO+pv&ABGln9Cd9v5}l`30GK7xy~TV`YMbjQkKM+K_M5<X+=uus&+u{7s<sUMU^YvK=R6JAObdtu|e7qqq3E^'
            'S5r^2M-oU)$tj#N(=GwFx$p0YHTk<_NX}a{+ju24NxaAo3*uz&lb_mht5QGGcjkHen(o;j(VgXn{iWF>S;eCpGIb>yG!G6BoO5q_'
            'F`Zur3sPx)SUK7;kCsfx7I}1L+%LV2rRTQgZ+Bzd8e<)Ch|D+eTs7_VL-Q5Rov)8r_xB@)<ZoSSS*Yt@3pm9ev@_@7?CGWTLecBZ'
            'YG2v&JIzw_d*X5LbgZ(e@RQc*2(y}08(rTRvGEzF8u^z{-1UIo_Wu*PWqJI661VJj)1&E=r#7mQ<jL*u*Al*QhC5XWXjg3#xPK#y'
            'jz`NX0X{oU@h+&{Ec&eeAWmPu5jS$@QUX>P2s>3O)@*e$CZ5^1@uq#oj6BEAo{woTh_^$d?Vmo%+*)29uu7sGmE|q%KvKC1i<mej'
            'q0lhS*rp*i%=B+G5w&5nwWwt;^3=7AMydi@bVA{Dwq4;^m2YfD)XDAeS~e-IL<o-_2iGQ0dKH*+$ywQYnpL&uP{sAZsX)qlH|yH}'
            '`@jDuFkW8cyPo-Gi*Neoo40f|gV35m@|V>a!)a@>e9|^j$=8w}?mi}42CX|syZrD@%&VT<Ka(Pkmg9?P?Yj2ezt<SanvOQ6vOnOs'
            'l&(&47G%*X7kNRhv+N1GUcjN)1kfn<b{gS%qgY6GD#jgAP~LCWq8p0n4$BB?*SU(D0|59i@|Kk6z80{~fU#LAr7(+Dm^=_I-3$l~'
            '5F~D@Lg)S6&H`MPHPPZgzLdbuj?Noxwgm=e+<~BqwO||^pDXxID7NG;;aiC=0u^SjLUV;#QCa{C6`GHQQ4`zi1p~y?FYvGL@UJ2M'
            '^)PVq(Q?S<nD9j3bYc-j%yGdj_NASucg!S6BXz&cxL|4ntZA(jy2W7q2TgrKbbo!l6yHiy_&vohxtqaXAJ^bpWkx?%b*&xYujrro'
            'j%zt2o|^O6=F)frG<FM~PLI{A*&ln(%eHaQw>x$;B#9xHuuMquq9+oFfT;TD7oWhG4=Rl|n3aG0UQwsxqX$R3cz#`Dqig)`1iczm'
            'qFyar`)@yN|Is&ADS+RcAc5qoqI6EmlbyZt5M(dm-2uYAA4`wJdr6juHe0u5S+!eNXSM(68?ZP(pK`3)RmHv;%RCJ{Zrr~jYKTXt'
            'r|40Pq9#WvP>2D=jx%my9^Q4nR4N6`o*bu?^@RXEjWDPN0ia)W98prwm=6)_upn0*&gH}w7rrN!a;nya$mf!i<=jmOgG31Sz>Y5S'
            'WX9F@)#jCF7LeUdqdSRb`y!st<0;3t=krEWvy6&}?t_gkIN|FBt6;?AX#KdZ4k$`?l~WX-3?ZZ*Sh^D4T$rk*i!Cr%fsuYPP;b-_'
            'MbkZ*0lUFoM)t_yiDcSw#S&Ow#l3$3&OH;eXmx=M{3c(j$gTzF=rI+Y!PzORa-V1O{M@{2i{GPP+)!NJ9UL8fZG4>vKfxY<9hd(y'
            '6y)I35Q+ZQSME=4xs6p0T8q+n_B~8ZTs+EVr;BKoUkw4XtXolkEu!@)dgctpiTU1qKKteF*MbU_)_9(gGJPmsMtHd`UW}6IC)P*('
            'R3t!BHjfGaLUBqQsPW=eT_J+1pN*6jd=D~;di{N#g1Gz~*zIH*il31U###Um&g*3iK;RKa`!Dj}Kf%AeKSS{d14?bsG$k}1-2<pV'
            '^8@9T*calpf<O&lsE60;;g?W6l{Ri`VdOkMBE<d`S_!W;ecpz&Seokg;e}1R;Dt*Jw_#v_XNrh_!hinZn+Njo;5a;@|AAaU|6TJD'
            'ZPTg#@M4Q)!h7)Fi3Ju89{4}InMv{wojdMChNncyyom53+MtAT%36=7sN|5-%qX&JLW!V+;!UjYmE+S}5>CtIf(Zl4L+?ZWVe9;P'
            'qxbD3QL?ykPJ1f3w{b}47M@b0Y6r)f%()pfV~F1q^X3D~mhSh}m<BhtTcHYP8cY{+41{TJPK4WJ`O$5bXAZdYZq~egsaFSlD-2Z8'
            'Vz-##q}vAirFcnB0f5k=y2@?o2#z+2GZ`q|xRQbZ_b=fn_dvrm`J85WElN7j(#=ETdZj;O$eod`3VX{~=m7?Dd44Gm+j)cxk2*RJ'
            'IIF~TTCeJqS@1H36CUr>!@H&idwvZhW}5P#o0OecGa0t0@Blv?f1>Wwt9kV=v8jaZ?x3Tr<0!SUN40Lsu4+TyZfj}-sH3&k&P)Sh'
            '?s^oPtob~SO>UKKtDK{`q!A-aN1|Kw&U7;h^zx?%|AgC9q9lz@u5%t-U%!1jZJ!VS{PykSc)PvU-Q5N^0&z|q3Bq5mJxV`TbdO2*'
            '%%LBIx~}5E2H~mEJyrAlCrOm%7icVn)6CTolBHnVdXMIUZRopl(m2HB`5Im~o%Y@@d6R#f<Q2b`L+R691E>jWJIwj7I+Xk{=J^VS'
            'CZ$K~0`<-OCR^#a@0WrY^b3U0y7Je{N7(ln8W*>CE}>X_^d*M#+uUC`zv=42Mfa_R3-iLlv*GdmbLTb|83Mry=iVZ(#;csIeV8Xi'
            '-C9`2DF^WINBx>=$&e_zqyrC{BALZL+xJfiM)S&hPB1qSDhA_t$tpsXEeE$NC1W32*J&%jp^hB_apuZsI^>s87NwgqDaQ!H{J_^~'
            'lYN5{pYCaOS$xG8vNiGc(?wp!nXGL@B&{fk8}?aw<kGX!DuwD6K9YOrE*}+f{|v_aGOwPY{#|ET%g9_fms!=M&a32`{e%qrnI+{?'
            '92{`gvlSoJAYV??I4YRA%PH$Sr%(x{Lt!K%i_Pq7%uPm<BB2b{hWmk5xytp#EHQ}64l>A?J-*$C+<0UZue}g>HZ^d#l3hY>`1Vry'
            's<ultSw>(9D5h!TBB<Hbu`Vau1&qH%Em=zSm37+V8{2FZbO7AYxKm74rkt;M8;MG-3M8Ttz&!&Eo5=x+(Hc{z=D?tqzOIMuaT7SV'
            'm!Ug63HN;H+BTDtZcNI6*3d{o46d)Fnzi4*;hyFBK1kF_nxLvB62Udi$u<D)Ws{Uq3R~|YNr&Q^Kz~te2PO^=Fg+cd10U+8K!z0H'
            '6oY+BVbW8p^Q|#|JM<^>eS2cjz{yYBzGy8<1>A7&TL0c+&ae%1hI;Mx#Ko6yJxj5(XD)!=wetqrZ?8D_vGu;)SJm0>BY^Z_#3OW<'
            '>HDsQ+(msG3gLl9sg1AHosO8I$0kg=L-zYm>4basI1PJ9n{K@Q?Nd77JLha#X>`X+YRt1Q=o_nL7e2EKo`2Gk?FLCsAN0a^A1C8>'
            '>j1S`;^1YQAISXNK!yo4khjRQ<OAM+D3-vmC3MnODj_IPhgmWZwv-U;Kz7_1<ucBaED@MeTSS`(MD9UgnERV-xy}TQ7=Mh+PF*a9'
            'a@oO6si`=|vZ<FwY`11ZD55(rl0Vb62+L@bMN63|&fN+?JdL&7iZkhYVUVo@P2<XQbFX;@W#i-RhS#D!t-}R|+bYyCZ1?6H>`?MS'
            'E_ly}9Umejf!8fIgo+r4`nC-4I>lvD;I)rQ2{Y&1EfY91sm$UY>Uz+k&1z~JY0KUeyJnEWe?NWB=0SUJ^vn&bCJA^)++fWd075RA'
            'qyN*ZKf0(CSed*&mBTjsYFtiZ->0Pgf^=ke&&X-!mvH#qm?;&d`sep%dFrj<q-xjDgCT)xmK6F~-;XE!cc?Wm`E~#!E0Z#nsxL-r'
            'CS?=dvuyvcM%<jV@3Dibo{+gTRWz6IHT88N*DCtEqtN_#3ukRz`XfBEbsgIcn^JL9d0;0<=EQDfeo%!Bdo)`UJyqy_cn7Zj=-8G!'
            '{J*Wz%~ak{+Hdx~p_0$mdqWk+LT#6}N*Olm8)2I0&{6m{ssQI=C|^=@avP)D3na^+eE)d2u_gtxtG;^u{14AxK6=5<e#E_KZ$LB8'
            'R#GbQ8b}&m@Q8k82Z}B<7ldk)?o6uHNC#04#B)}I&+?d+_LTk{DCCP5KcWcy=FKUN3ddax$RXeDIbAG=cf!^c-vklL^wV*32@4M*'
            'DGdiltZ(MC)v*HXx6gO4%cOIAraqDy<gVzekQlzq^~}C?QitFAY}=+eFByj~$?J{mtB!QC5mK^@U)E{*ll<J191pqw50uA0V~9F>'
            'r6jLuC?|r|32s!efzEX2Qr%uBJq}y69trYa>ZFLrl2jVr`da#O`?6EW`jN2`4%1|n(A~-fCrfvsY?a8QSz%b{CB+}uy_Px%n_;HH'
            'QWu~GG}$N+*NUsAQs+apS199ouDtl&$v6v)H^KSV9>40KypqLfbjO@H3>V4kF@<6+N7okYZCf};7X)A(H~G@Nd>f`ZzQi4bHPfiT'
            'DW*%zdQR8+TlXG5XD;=e>ndkfx6b#B-9(qV%bat~o4!+Guv)_3y>*1XEh;92w0PruY259@n+e_cFJh9vD4l#lHvt5HNPq&RK>cR0'
            'dunus-sFYS|9oPM;wN)ilA7C$p<iX($yv3u#=G>z<|d>=dW<$Y*H6>Td}}Ic`<!6B-D^X?MFo6vup+tZDo__d-J%v=?W%$(bq#4)'
            'v$zd(RjJd;m_O@t&-xYGgSWxS+v?@*EN%mo#tpv*I_2^XE=+Cg)1SY8^Zd<=r{iCLcz*iR)7Rr4Up;yH;`G)VJiv?L>3&(hf>s|`'
            '$~;X?QmW|g&U{g9A3CD;JZNsMRXe?-ul;S%x_R?2HFj$@Zt;}TaN2&4%Sim74*eMb3#o6oA^~>?RK-mW&D?5?HBQrht$Krmy1d3M'
            'Z-?~#0hb-aY_~6C!@3#nv5vEFlq*_L_H~moNVgi}3xd>I<J^q75QQF_YJ<TXcm~QN-7A8#=-|Wo^$EJr@0joY@OE>s1H^4+AyC$*'
            'Znad`#@KInhY>_v#+yha%R~VGJx>!cffGh;7B~R#lA&7UFd^{GJp+4rk;Tg)##Z5xQNj7}5as$T%0}0%7GYbtXJNB`lAvJBAmgP&'
            'Wn8pu%D3=x&sqm9MHzIed51<$yP>l{e+|w(pkx#{x@)yWuWoSoRrxt+c%~3$^v;#dyinajZeJT&&46qRa9+4}5f`c!=bRa6-uqzc'
            'udNwq5{<Rt@%t`#^wk58MZ*?2`PvVSKem81PAele*LzX2G3tlxyxe!d*am#<yyFGt_yC0D)oh=-gie^r5I0Jk;oZ0P2~H(?z9ZRG'
            '28lyKm{|sm9@tXYG)WlfmZmXX2U3OV=d_sXXS#+S=@Yb3L>S8lJ;!*Tqrp87rTPefC8<rs@dNlz=5>%mP!}p-V8jVWVwlP*@s$Ou'
            'FAfjI3A{Y9cx8R9LK}+w)D32@@qRbV#r6d2&x6(aZea4<n8L9aC}~7W0?W8eQiIL*UUMCquFeN^J1jGWih1svcih-aX>d@P#T9>T'
            '`luYNqBYX;qv7T7h_1`gFQVbmcc$0r!E{KMjit%t{mK2iRG1ePA2EZsL<J~W&@E#p3~`;jAJ13xWm8|?JvmxCU?cn#MN`>XdzKY%'
            'YiMtHQbW%z#v^;6&7$diC}sSyNl~2cu&Yf}I!a%%iEVmPiLooEFUPmbTeGvKd!C%cmQA+Z=I+~;6t+o&%MwOVp?atk(K2DT`iDZ9'
            'We(_lT5X~a2{O$!Dw>iqT16iQie%VXRYv00@{Ds*n^LG&njs3jQZ}Lzxu#6V^vZt8ZDNHIJISfqF}kg^A$xWxOJjO*LUp?}&JFT<'
            ';&pe(8BIMMwA;--hI<_;7koUCqeCJ+A*R>ci68E4tJTNcdT`2A2^wIUdkOVQQPMjv{cAL!P=x89o$oL7K(VlbFV*DmJxXq8Jn}`F'
            '3Ku*vL(N^9*SgwP532-{P$cL9VWZ5pukag4ScN*;@kJ4>7do6?zvdJh^s#4{RzmBH3hl$jojdf0+&y;+V+Y8OvF_JKVdWoXUZG>z'
            'B$(E;?GdD<cA`}~VEbe2_FCB)LeJ3bm1)9iM&^NS1m$f;))^e`wZi6*IUlt-{IChC8?~8s8;;`oMlIga3~p*OzAh9M@ziE%?UF9P'
            'KJrQdb{(4+A9;9tyS!#aM{OR~NfKc3nWGL~)tj`Xp75$lk8C_O`H9~miRFFKYh!|u<t!(CLsGa>8F98MSC}jlc<PPwE}Qz0%0Q7('
            'aNW@}4W+*g_9+*;?v8npaBJjDCKNT8@1M~f+Ou<9o#47PHQV6Y?eBJo_f_JYqx5}tnb+0%e!B@ylSVc{wdu{xNgGhd1Df|tI9B;j'
            'SuC3lYcP_MxYHeAu-Pf@xa*ctcN<#qXQL)4G_(TGMn+_nt9CPj#YT5jwb4l^1DA|e)&AXn<l^a(`<D>ViQVR~2eJhH*ao*)K7H=('
            'T)yTW3|@`#lx}W*j}RUzWwU)6h0c)96;DRoAm4Q8KUlZ~1+TWm63K)y>kAy%Jb02UV~i`E715u?*ZT0R;w74w5?~a_$<UDK==`K('
            'hz)GWVW4k0x&1eevPk)G?DTqofAwxH^&}mF0erG25QkvB@JKqmiY-y+){>4R&MIMtE-lfbZU1BC9_UPdIZvYtVuZg&NlNB=C&;a3'
            '{>@U6M3NL;X{H<L6KpPy^6H|6+w)+Jw24al(c@%U5uRiT`L`Aza(GNhve~9C+FG4!#BR!s8Md*0Cl6-$%s97EjE)DMG$kWW)oyw<'
            'eI-spag6uWo2+(gFlTB~Xwvv*4ERB_n&nq?;r$0-+Xi(tMMuV*DMzmp99{rb^Mz7@)YeEF@+@GZK&y(p|KeO8ZJ3c5e(r8yEX_&h'
            '7QqkXYA%7k7dc+Djx?Z39!lrE%UIKgBr79~wVTE$G+@%Dc*&{_(*yga-Zm|xR+==`vuF{4WcDGA6mIcV#wf1Je$_^Xr#zk2!Jgsn'
            'Xrt^3aB&{9l0rIrs8^^E|Bw_mwnl68t-!QHDv2IyE{zbOmJ=4&p!-KUOzXU&QX!y9hvuYD7vw5p%+pw3YO-SrRj3e)>bUB5x6eoe'
            'j>-pSo^uNI@eFZ*>1Rrg<U>(1U!EIh4G-Y+AS9iPO-8lE$zf`wU6U@uO=`A&znz4~#5KdyzXO2d`)|ZRM{xF{CC2IA#@Zb+<6OP!'
            'wMWs1gIA|(wZCApxIH(9(VhbPK36Uc@*A$uJi%PzvS&n_I{IYx`uC`qy+_k!zLvA*^v@BiKj%0)!tCUZ7+Uxhy5ALv8oI{8+)!fx'
            'NdR_YRvV-VP9i25Kuj+3RSg8<p`TpEnN6@hmS|bMvGi#&Z>pkT@>pC&Gw1|5?Vy^);9`xj!}bH24iI!kyWh_486+8;e>ijQ-F}YK'
            'C|_6Gb6-123j?o5sjTb$<^u4#4GnFVp3v*k?l^}Kp|jmZywwa>3v8$L(zsL5m^MZ7+7s8%K=|gNj7~3FR2Grgedq53?d6Mh50q!1'
            'nuegmwb)HI6c`Gr=9uuzoN4sIOR|igPDxW;gXu9%?TxpjzCC(}J|{5ww5Fpzs1^^`6!k*gp>;*XxQ$NMhNG?V+J;X(=(OkG0uJk3'
            '6l!075)>KzIP7#A=z?G?k;gT26J+M}DU{uvZRr>BS{U<pM_FZxZ067JH`L53y;qD(Cp<lmUEt*G-WnQH8GXL&0Be=p4O!ZGQF~k_'
            '7F=zKc8hcYY#bDZ9zTa&l_dV(q#U&?Nvgj~O(%o4b~ddrwF{04<m~{`a(EbuTc}V2zw3ipxl5ZGw458vS{(}Epyk2fR`TeE1J_T%'
            't|Qgb{wiipft$3xx>76Ez>NxE#mksS4{1D#OAjN$hX8ODwRt<M{n=8RKp9(&;w_7cm7(b`;5|v36R~xewyC9;NKWh6%H+WICc6w@'
            'YhV<T_c=HX7L~~~d6cY91*Iu(RnRJQ(~+`yZCn<uxonp?;Tx)ONl5X_$EP_NT*9g5S6bptncffXcnpgX?J;Cwl=%rA_53Q)t}*zm'
            'Q$|<J*KS8TL^$N@IzBlNIwdYdfUa*X22q)m8>9HpF*JLTnpInEjAjwZNM$hGsUGy!lSxF5?_*^np-3ADnM&NZw?I8ZGw9J+_}M%|'
            '5vkLzw?BpAr0<`xMZJoY`o+iE8;Tiqd9->po^0ez@8-*>QZH>l=*m-i%9GDD9(**Vt-2QT4;)0Lnz)^U(9Q#cI&<gp*qd{wF#fe('
            'U*^d)Mn8D@r8^gAD;Cj(?Y`BZJ$7MFE~AeNTI>#1>~5z!ig8!;;;f!7;dC6wl<8z<f9-C}IX;Q+xC9;&1N@pSrg!d{*Dkr=Q9s??'
            '0(Ok)PE+H-?r6)MQI_2;-75U@c0qXEjoc8O_X8~2K^k9T8WOx)FO3giA4q%wHlZpuVx41e^d-wwb>~IPM@Qog5Vv{~lau%=l8hPN'
            'Pr${119=bRdn)hHwSoh6LLVd6E>!nD$7OSHa}XNVg{`0NS=U*qy8w%rER-b%)hXkj_*zUTCr|_FYq3h+qjHI3_Bc%sM!>PPgs#1Y'
            '&7ri&=|<aEIOL8=c}SyIH~rZ$d}E~(=cp(oY&d>e4H?_WDFRtDMNYfxdXgsP0{fM1DoX6vGnH2qh9oZ->`NezB(c^l3^~ovyv|6+'
            'BpyF{`SR5p#0$W5Oe2Siz)?fWF)>k6!LDA$yf|5|wKf9KyC$zR@d#!=E*CHpI!2KV7?_2ZuRLt&vI$%d-PfePny(<z(<%%L7M4JT'
            '=p>#-z-~b=Ci!fG{(9mH;|id?grhh&ApD4*02Oko2n%ftW<j~uO3qds%>e<gDCOd15>e%MX^vI_jl^oC@&OPSaApqE*z3OwWu7ZJ'
            '9$xv=A08dR(WLSc<13yA&V~g)=@3QVm6{L-U38%Hk6u0z5#bdst8WU7`=IKzbZyiN0hHCSL#T__)VM>TPl@IoyJM;M2bk1MP>6;c'
            '3B%}9ENWRLX)0pOxkGuS7_X=3M=%pLt~bw~p01`d@5G_;zNNW)hZIe2cnXz^Eu#>$9Ip`t)rIXmd+=f!QSwQh`T1%QQPNG=dT5fE'
            '(G6N=nCq9~)mq-Loli)}Dmc_w8m{_>(lf3<0Jef}gD+`C&_F}Gjj}(-l?10+#%}54(vx@*;ZyODW0GNrc^VTKN*axOaEUjEH$lB$'
            'D)I_5j@dMEWMoF7S;4-cF1?DQ8Rex$%q<uB6?I6I>m&n8IQVRly#MWkBH2#@wf34*9m6KiV7`n*on{m)W4dEH{K7OH8ab}~fSc=$'
            'U8qFf6;-|7+2%G7J=G*<Ve0SqTfNm=_H%c*0o>8&j+Xu1w$ESlKHyR3SeinC<!9>!gEm(9MkwmNQ`48ks4LMBMlqKuOh{6|#+&8k'
            'hHf>YJVC2yhQW=#HZcEKE9zSsX5{5_{uQ^&TAB8I2A+0)CBvKD2pn(cf#T||JIY#-ZGOC!!P;(r2h=3RaPN0VtQ*1V9H;(CA?oHZ'
            'eru$<cZ^Qhy3yWWD|&tK8^YI(?#U>2y7Fk1lpJ0;Phw0gU%{bMpkaPQDU_QjihzaMg-ukyA9~Y?@i)^T9ho=cP5f8kNn>>5k7vA$'
            'CZ$RY-Q0gO>Em@UdIWn0g6VOx6T_VFYEIVy`Vu6z@;qXnjk)UUv!DfjcSb-D9tE>#nNXGp%3Ww)uK?JPcu)!0f_hVzs~XO`p~~=J'
            'bfF<F|NUb8{}KN7Hwu6IV-tuZrpXd8?;p``e`nEew+MW@Ip&Se9o8tKid@~bSXwb>7SGZYW`5;!N2Wc>FyYY(9d?vc4~Iw@2z<+c'
            'wC2)CAJ-Oo2fiyB?H|!upCLNyw}wr0giQPu!zF%0sDzC%f3*+^MrwPt_?@{_ZCcV&xQB+KJKUjGZ!6OKzb(ijG{^9t9C7hA`7{2e'
            'xQpuO^T%7LPhP3kJl8%)V8&U*Z~lu$W>{I8{x%UB)zLp9GyVrhX2_{O<mEVMuCvw=8wrPOh#ogSs!;Ly5U8<u{_^zEOY!REqdz?t'
            'PoDna`HLsdUt9OKk){cFy%Z$r59l4e7I_wJ0BbNg5XNbK_SlXa#`9N<^l&ZNWO8U-r*cc~6B$dUI6Za6Ouu~*3hUQ@R`wFxOoFU('
            'Bwjpv^YpbgI1IG2gmyYWx&-J}N?@4lY(ln2G$Bn&x+^78I($;z=OWFsi^kZ3IY8rdju+RGb%&Cbrl|y2q2)r_-_VLhc|Gx(6G{Rq'
            'jVz9l#4(Wcq<KU_WFraQD5IKF>(bYD6J;-9(@x83a${7EToA+MW4AuND-@^kU+{tIQg+`A4Y6u0H7R557G!G#Z-Hrs(Zr5D$7Hl!'
            'y_A}$2Ftm}KK%DaB>dM5Pck*~JNjT;cjUw}nZ(%rE@?0pb0at!g7lDxBk*x+%&jtuVT-ZOK1rMK4qG#HQ#{oTo$2}fs4y#fv`TTi'
            'U=U2JzcPT2iU0GLkDtDH`s35n=i>XPe|+)$Pp@9PR}S<zcBZByU<RB7CRxn0BI4mjn}Sow!FC4eI24^mY0@bI8sfDleP75n`kJ|D'
            'VQXuB)9^JWj30<s*qJ<=Vk{3xnRG^NfSN4xvQAm!-2(2Ug#Qf@NuG}Vtzvk59$Ct{)bpvcy6w8<La@JnY#5N8FZgd15Y$Y5+W;91'
            'yG)b+{Q`=*<AOd{EYRnR|M|F>A4f}ou9S}6V5d8~p*Vo5JG2Up`=_H~fV%FhkMex~E#B_k4(?q}?YD7J|NHSn{td3hALYmVKhh6*'
            'nxgxW<)@=vHDBfj`C7X&X*a4&x-Ri02Dpgwu6sb!yhcYkx(ZO6j#-vN@gx+>coS+JG6-1c9!dsXF&GSV4c1U|4c0VO@k}aQLh+SS'
            'QwO&IMK&vjB+C+xJEi|W>JI^e'
        ),
    },
    'video_analysis': {
        'sha256': '57349f8d512413efa78af89f6bfb81aa66a92b04d2b91f42800b68e7308efc93',
        'encoding': 'zlib+b85',
        'data': (
            'c-pmnYm?iylIVB+3LK^C4jD_b#)-3Ml_<A-?8%916FU_<$vt=O6hn#7h!I6<`IxahJ5~G3_RFpNzivMO5CBQdB%9-^WM)X9(P%XK'
            'g+_Bc9{(ttDqdw$+@y&trg0u;$11JFQ?V`T>Au*@ycUV9<+@IbTx7*{x)#+oE@dKC$I<*eqK=X77(qAgHA@f2d?n&M5%YO;ESkDm'
            '$tZ5>Vjm-{XdhR#EV<ciG<s9ia#a*NF+UeODG!xM>q->2x!5;Zole&PDNW*9ij~aQ+kIT_()?Qd^jZY#Vt<HB{Hu~TvW&CQFWa;_'
            '$WrW!L}sE&Kgwz%s<q6enoLo`1H`R3ATn_#zWeTbXi=21te`^z!yM^nI2o;)TGU%9;&t6X%Q(9(%Cz3@5ycjpN=&ED7yuW>Xp*#;'
            '9%Q^59nv+T%;O@wtXKm$o?ew+BNi6b5gDA*rYNCFjL@_6MvAR0rKpP0s;IXjS5&J6fXWD}`~Y~0D&9-6hUtqm2Rg?v(~a22X<oyB'
            '2sUXBS-J*#PDW{-$b*Fc<#l$PCS|&*vBkPLq_Tpgc@NY?QZ~RRIZ{|a-y%+6c}j6;R#{qYWd+kZ)?4WKC6cAcv!f^_Hj33miBm?K'
            '6rdmlZh?l-i{Bked6UXpvCh&1avx5llyy_)z-Vbc-OGJZ9tEs_EULO}U?oa98@+@{<AMc-4xs?vgs#=0$g31N8|X)4;go>95tFx?'
            '{Z;`(*M&&ZYBw8=$K%naEcPObHVp!gB9ZP7MOg!~0G)WY8jaMmQtIETW_2iwwX7=rq)zveVIAUno24ra=m+?dUmzc2GyO*YJ)zDY'
            'i`;<U#ub9le<IQbiM-h#jv}r^ei)73y?^okr+3lMFTVTf;+<HEyODr@WBCF0L5eeuv++cX6Ij2jIN*cmki~1cEwV(G_==YxD&@a7'
            'DeUj@gdxNSnBNWlEylo0&1RFXQ|yk5fe$w^EGfN)hDk$=AK}s;0mMCE_R|kPy#4Y0#jEJu+n;`Xc@h2m`qjnT=;D99zj*WR_1iZV'
            'wzK`=a|ATo7dQNK$3Ji7YEQq!Hz~jP{Dz@_^Y$f7`^SsFD@eh(s^hZ8M*B2LGKs$dbK{S7Q-SyyhojLbksIWaBD;~T@G9f{S_Yqj'
            's3@9pEu*9<SyokYJpo37{xGp?c(Qy0Ys3T^_=lX_0@$aji(nigLb84QAS7-Ncr51=4>mr0_W3*%)2~GhJ5ye6V0YGlF#fn&FlkZN'
            '({y1!Zkv6aPd6njV*=YWEAs0p?M}h7!)*cbw#ZZ5n*hJa6BGmp=;A|K(O%M9jwBCmLX_ILann+DnXcjrwr^ZsH@xM<d)U@5fi*z>'
            '60n!BrwZ7kFqn~36(A*k{ne)SufK{z)_@F3^9E!F!=c3zAerLqHa=D(CRMyYz>t8yC9rul;fEM!pmfCfTtgb;$S@HYRPJEA!pn=d'
            '-^?_{=x4fdRvz#a0Walv94^el(E<Vg5zzy8%}`FxL(^2RoZH%7Pwaq8YPtXtJiu3CjxP;}ua;uIaJZ+8(@Kh;<E)VvWm%NL*hA)P'
            '=puj*x}$!Lz_WdvrQ>#J7Q;OJ#0=a^8l)tU$JvrCavj7r<5ila`CcAn?f_bdJNwVSl=ljsx;$DGcQ>h_X3yreXY=ok1V&4u2UY<%'
            'Bov>DFXpp3b(SC2@=%N5{qex)PpqkgD21WLiOG(6gH@nfsuanOQYZbWA5`%%t$<fG`330-0I3P>&(>Y|?r_b=CTEo^iX^^H1xW2&'
            '?BZh{?}UR3FG6}ON=4;#(?_O<26X3>#JhAC!*t>j#+#odu)|l)vDvu{W-S5`++Vs$4<2DJ=Pj2)$^Gt*G%J|(^-@Zr+s#KtObxan'
            'SSZg<LPdqs1vv~+fvRNN$>H@L3FGMzhgtss!}c#~@e%B25aSsLTd@3jRUc$mUt;~yQF3_=)Y%{x?c<M@Owyzvo?*eXk3R%3{mBU;'
            'hYUaow6bXOBxt)Km~bLKga7zd%Y~n5iy<Q1qjVF2-dIb}dr1IB&o!`oMfw*N1J8E7mFpdt%1d!A!FsGq(-2Lq4j3>`u&?26)xD*h'
            'dJwdzG<l>f&xl_p!Fr!egh8m){2&TqfRJX1P+i3B77g)jk+YY9>T=E6_zawhq5!iIW)E@-*z5p27hlh`Mnn+E&BpSb{Q%8#9=)m)'
            '@b#AV%hx|#&~sUq-RHIZP%qz?jqH)|j3c(69f~XqLR9xHiS@setKzgo>v~pYV5q>NpWFO|NNXfvZZszNmHQ?qCn@Oa!*(Js!T-#_'
            '<x76x2WA=8Gj5E2*u)zow6)6Lz)-rNUs3y+p$G&x$-w3=4jtJ*EpOwr4seRN0H1}vB(N}k&y6lV9MID50kun!+%6&m<Urbc2DRFP'
            'q-6s8k0?n?vqSj_`bbN#q;U73gVZrsku%f$06sWtty^%t0e8WE3i56&(5cMP<Awc>4l1~wO|A^q7x3r<`nuqi{`zZYY_KhF!Sn=P'
            'lenk{XepDKcniapw<xG^{9N9G79wz96;I{<P#>u=xM5&!GGd@!J@_zN01rG{$xN<mg!|rwR-#W04KQMW%Z#@5S_FIL3?fc?ZXpK_'
            'w6%>U^8uz$p$SW_0XjG>v6A5UvXz0pO~<C~5Mp8Riz3&ORwU>&b?j0!`gD&OI8lgi5cY}uZ{W?FqW%V@tF{3D%{E9@P1e9v<0J(O'
            'P{X_*JHwfwKw#HF2tOi>?>xp3_D3tKfnn4Z*RPn5v|dTeUb?_@BfQC@FwK!D@WZCbBk8dNccmyBmo6<}M45Gsj*V4-+GV?Ikq<EZ'
            'LB9Zd+BgBf5mGj7v0b}_<_)qPy&~uT!z?RqQ4t};2Yve<Anv+>&p~ajY~r;OvWzIR-N3%W5WsH*S&aB7co2^;e31AK^K8a8p~-ko'
            'q-#F6t1=U>&`90EMuiRAe2nut1;ZB?OCx<hHsIBwDJDY$q@C^ryy)W+%$-#d6S)>5@Tf+zA<CgCy+F~RR|F<G`b4}#K`8eH>XPYg'
            'Q6^$5o0269SbZqZK<8hl7;nt2sH+Y*q6VFfF1~;J9taox^7hA9R)nV77V907pMHVn`1?xMWqfRZ<4_ao5dD+mLkW2PTlI^HFWtrz'
            'ecGhgTZ-C%y9T7e-|DjA<`i_qU*c6kx)5fjyJy7>g?P3=J{^q$7lgu5NEI3mLk0BPDk};`rHro;2#m5V!J^RM(rujc=v6`UEV5?L'
            'KfxZP=kZNk;V;+e$Bcqe0DCQ|F`BR%QZ2Vxx+6q(F^wdD1I53km{}d;WHEk4iW6+0GG5maxuC!d5rvI|&1Qd)*U>5kkGQHqbC&U~'
            'C59)=h!JQ*^*42E|2`E}5+)$95Gr0|d(7scHl!#{<=MEPPF^;3B`yuDNelLg<A4iP*~P}20f>;g5V2CMCHfK{VS|4=d-lcWPhY%v'
            'Nihng2NjJlo_zkr?%R)I54NlL<;BnJd(^-H)SQ5!)Pv@EwJuXKs-AKn-l#+IBT9ivfN6~Ti6j7-7SOH9(Z)$e$}&WY5bO$G459eT'
            '7{l%oXw~38W!d8{j&A8b4SQhD4snSGD*GJ@Z2qGnlX(I}fqfknJIjK00sfB8BNJ={`_)k-Egu5wPHCf0w}4}`iu0T|)-)@wGmI@0'
            'qa!uPE&Gk98+0IO*F3qF`Jtv?r?`87uLGAULc3#CsWyhMLhM<qrw)(V{r2$zhnO#J7IPZQJct+b|LNd%v!QJbhLv2>;s@#N(`VtQ'
            '&lc*hE&b*6SuZFqou#V}z^XLB0d@&X4m3bFaaHr<^h31S)9=UT=JRLsg9nJ@K^(8Cb$nB#NyL6H7}89qeY}&=M_Cp;k>aK$CJv+G'
            '-Ig;``usPsRBS2f@L*|nHfetIdqoNu_c+BPMD(V~vLDoIWfUj}u-^q$3^Y*#8KcPahs0lc;z{;y1Y<~BLFp<LAcrgH+hp1fB6z$#'
            'FQ=c)=kvvtF^G5%{P}FG8P+E+a#7x4Gp8$Sx$`doQC?|U@N@cIF~MQIkE-)8=*R*`@kE2Yh^JP-O#V+>0u0`vuw19RxJ(~h0a9sN'
            '`qH5<Dl<ll^%R}UlKn@D4K2C~gg;U}kvHkOeY`$2;~_nU=e|u1yX1IiJAg*{>v*E9t0>Jk#R4q-B7=9|#K6!)z+63N^xJzgjY_mL'
            'U`?^16YS|NA2x$Zp)o5FY+%bo7}$Vyl-G3j!E10!hqp3bZ}kh@aCc`$xSdVJnFRf>enlAgGrqZwfHMKd)frFVEH5G$Rj#)YKNV+a'
            '>Q!e8ap}No!BGSFf$d`q`qsy@`>T7`GbCNg6N}Wu^dY4>RR12=j<*>fuP>c9LA$nkF_-*f;w;$G{)4HbX%(S!G7-3T7~?~DvzDC#'
            'N6@uv0=p37hfRDf)lu6@#(P3!`5a88I6=P#AhY%}Q1AulMKGG-UA4R$gM|sbz?QcU5%$%1A)e3Y_Y;9+jBt+tmG_6r1BVA+DRn7u'
            'Yr@QnrXproGXL>`f0iPunbkS`v!t^oBg(!jV+|R8%UwFR`@b)YW(yD-_AEYN#z3I{R<S#nByK_Do5|}LJ_8v6Rv0*p3J)6x3PpJB'
            'oM}X5aoe4W4Z?~S;>x(~bszGB@aPT)qn(1sXqQ)BpaQVLmopA9t7xN8{bzVG*&BgW3ty*sBb^rpBTcZoPQS6)tiWL$dFBXziwK9M'
            'yFJy`?&}AyhI-+9=t9lZ7)17-rk1Ga8^n4|N1U@cjQP0-=)S`}CkQe~3|zJ$S`5$uyY!P`jNAsTea7=zSyHzLwDq=U)?-&b_w;ug'
            '^noBYRVzg_Yv12nCv~l&^8;9p3StPtqqi2BV^W}jKC3`*6YDJ<cH<hu53`-&&u;R9=@a!aPlv^r9s|QVAST3L{d{&mIyq})l5kp9'
            'L^)bjVEfTJDXcX+U^7E+4ohd?CSCSqR$SQ!ES31-_zoezB3L56jxnP_w8PA-Vb0V5XE>j;IEfEv@1L<Q)^<cpBcs<GSPr&g_K=a_'
            'jE`#O=x>@K52lqjq4qtvzt^Z*@wkRPuHH^FlO(6Rzfc&yQZFW$x`7kW57;2k{i1b-GWgEWapAW|pGU!jmp=IW8E{0Pp&4Fxz;^`k'
            '>HLB!-cn{>B6k3x0rOQu`%8H*f|ZKiErddOA~b%X_A1yNSz6Q!T4#Vv5T&`0lF^=t4ef2(2R%(@agszDo4}rA&<AnW!^d_rVw$Db'
            '2k05KuY*vp*G5DtzlTna&@Bj0oI$HC#=BN`VBiW$*g1FVIoRre(gHw(<TxM#RQuiynqCHGJgpSW0~Gnn)Zmk0(%QL2oGT>{v<0ec'
            'NEz=c$4jY;Lj(+gXDmx`KA+%mbCdu8G>@zx8NQu8KM`%)0wCb7YYgWywCCr~NVFvBx;9eeJIblFybLhEsu<4}DWi#^ULgh2nGBrs'
            'bazI!C>Z|vVU{OxS;j~32FiWs5zUYt1d1WESAY${s-4YdSNH0mgQD!Kqu^60*fTm;G{GdQyb$$mAqW~}TcR@r1oo+8WRj90;Kr9l'
            '74t#@J{9f_^>Bvyq(R4^ugWEP%*r<2S5}XWj425eo(9Jyz-y_zY<~?e=WGZPk?ihh@q=MrkSQMkp<+(|gMI<P40-sf90JVqKq}zg'
            'jnmAtQq$xEvK;}wMw$}T%f(`Peq|l;Vt+~nU0S=Ux`f_W&e;rj$3eWRg0{m{fc?I|9P70lUxk?4<u3yx11%978(q)zwO<+ItOc$$'
            'xEY^QCB`LrMz(PSPZ<4)_#(V_CV{kdP{3pv-ewLis}d8YJ6I7FtpgIA)uKxUTT<{05JmdgBD}}niHG4YdlVg$$D|rOoVh`TqKU<S'
            'z*^z5R<yBH1K20z))EaEg{-&*=1H%&W^K9Y4AWMSzubo~R%lcmb(<mx@s&82)AMH?>kd%BKm#$XLK0==b7%hQ4THkGrJ#izn$w2^'
            'o;h%0bhN^yjLTG3LHCG286W!ZlLri<xRGTBoCL0JWSlj&rcmx(J<y#6c8iYc%C^_UoNOB({c%VEnUx|0Gg0Q1d)@&n^~G%d5Ms(~'
            '2er*pvwF{GpFMw|`v$i1Dqin~0Q}4IfmGfJsf*U?yr6XhGm-8sB$&sL0Ke!23Ch$5Q+>bZZ2p}5bF%f)S|vxkqp9%WiCQi`Aylxy'
            '$V&5d)_@KWo3yNI&R}5ocpo46Xl4&x2^?kp<J@lJl5|!qav7I+m4YrtsKo?f&8a<fUje3okPrv~&=M43%)~D^KuvKoihgK-Ehv~j'
            '#tsc#1<)1SHkC}+pr#897>l8Z108^72?3H-J=KuJN^WBe*OW}8byL?mH405W_?;(Nigi|0bl+rK)cRyA&P8F8My`t@!P^ExU50UD'
            'Y<AM4PCsK&(BWw%5cGL%WD?(WKNKa(r?d`8hPorIxw8w=hNJxp2g=Ced+On48hF!CoBh)A=0P`2_1n(&v1~>!RMjU!oD@TTM(%<&'
            'T4+bgtgyKn8m8#s53_I6{k%!&=mJdP$v=AvdaR)ds9hY{36PimT7K#*l+uT~XUhVYHe0m4kT_Rr#arP^*{hUlaYqZ?QtQ*8%LXks'
            '>CNS^6OqVlJ7b%-qj5m=UY|CpAgY7SrRnbjvzh^U&6}^5F7>pfmVqY3wfXNs|9WXt_^KE2qau9IghCGUb|*?vUY*w4LH4Lvpl=o-'
            'CrS5Ak6@b2W0P^}u-3E(`>(VaGoV8)gEG$2>s%&=PdmUyUK&CT*vM;~&)BqexZ*hO$&vPlF%0NWhVkyxykU3(6~DuH)MlS|mSXDc'
            'u|Gg8-kM#=twW8{!9p*mDL$G?YU?|)&PMsk<H+P=F+ikG#4EYM`QBju<_)l3O!gn)`;>(ZqehYvO#m_<*cv6R%XW=|#umv;yoR<-'
            'RjZqbWHr>rBvEDq>S%3SXmf;haED`n8qTwcr)<Mf4+(+ItS1vk!E%Q^I1_YRacM<d$A<#^XpDxAdHz#H{%Mkyd}B1urbM9W&AEMi'
            '?Be7&WWMybgWY2rvHo6!ztczQ0cQ(2eVsR_JNv_0&;5?h=A&@5{lIEz^B=Uc|KwKp-86r1iQ6Q9XBqLaUO43&5er7vz0sgeE9!S@'
            'w4+HL=4l^h$WIt_ZM?gH=CKp08umYU3%UNrXX^GtiHQ(yOly9^6Z&ttQGeKhdf0V3iLf%oA|k5Y@7aldX&#Z|5W#HmRJ=a`_}TN>'
            'yge*%{j`^ui2b<B-sWz(2vWR{O&2ivHX|SV6PPK1VJOp^XJ^Q?TW~n+Y)<Q5x;~1q@4BmaG2M3$XX0hCU!^(N21+g`xTN(_G4K>z'
            'q4p6)=wRE)Qg1jX+0?XaHwtQ*4bBIX08qGR@&nB*0p(C)YdaC}R$?kBIGpq8`Fzeu3d}>OvJ@9xv9U@Eye_3oQqUWmV-IfM0SLw+'
            'Kim~+3>X+Q_bp>w&rL}H1yx^j?ViEd1rbslD1iCxCay+7rg*A^-o#ED@4n@UVWvu4p(bw0#fcJN_~)qIp%JVQ!FpA|3cY#zUR)#f'
            'u@-`RZypN}NNL_i>Mf;}Rzc``p~Oj%nspnx)G|Uld2L{nEqhdbzIX!Drxh*gobCz!n4jCC#RRu7V&#lOaX>2-sTT>1wOs;G?VOkp'
            'KSepgG<SL}Tzf^XwW;hn%YbK#de{?HYVDprN6zsg)e*OyqtK%){Cop=ZuywvYimxpRp_<lM~koYgbbiI)1fB=<B|`%uWYP(XQENz'
            '=H`^wNv^>8mU6ZMieWsu9ACbe{_lAD@&Eqk^#A?W+0~OV-{&&<=x$~Pp5}HbhOaK2OoJ;o6aO^cowhpD1JBsWr{-hy2_}2==6X;g'
            'd;G5R?NXVKDE4u+!)u~^umJvF8-Nl2Lq9f)75co*hZz_vZ^2!4u83>Kg<YrFcq**J;PaVvN!;1dzc;_aiXJ^OzoHEkb@M9>+hgDz'
            '&*R<qeG3UK`yTh?M1XbK(3vvH5nNq=d}UxLRphi3PaWd=9m%9=`&G@}k`-TJOt4*3VmEx!6Tx_39A0zQp<`u5;2G2buM*(x5gD|6'
            'G7<ke4?|CI;$f1B4*2WE`q-O?-Wcx94tvWO3L(E)#1-em3mmn&-JZx^#@wBEJ)>_)fdR~wVGtR^0ALa(m8=yu{rKVaXB6+wLO&yw'
            'g?q%woEholQC6`#d32?cyE+q}D<&QOvYiu;^phc4P5LHlr}$d;A;5<OBp2pL>hksy{$`PWiwo!M{40*HK?{84&hy@~uqxL#^;i<@'
            '@cE>>A;?zl*<kb8`STNGPz{C|2VT!7{p$~l_}q5v=sYt|<cEnlrO07RHhVI90*4k>l*PBH3{`EJ))Jj4d`aV6A#;H#IuXwvByB6<'
            'FUS6fpoEtlG&;TLF*RkxXzTKV@v48?fzfh<!c4Uf5#u(~!*PFLr)<E|EU}?Qr4)UfN|pQa&Ka<WWvZyoPJn!>w&unS{+p)ufq)fw'
            '+hS<B((xE?96YQX1ZcPF8N)v_b*9I1L=Qf}W7dDQ$fX1=yL|zkGKZZr*nkh8ZTswrI*_)g!qp;1oC;zh32wSTTi=W>YEr1(VP)rP'
            'pndYK7zRBF@^tt*kF}(g+D@2bfM(mM=a5!0g~0-RkZn=PJOp*u#xb47t~Es`k*g|jsb=7Xt&gBzG5oOOcz<mor>9)+uw(c!KqQ|<'
            'Wb4+YW%sotaq|wmFp(1+(h*i2+X7+^9-McRgEbvIF1DVx`)auN>`)wne(Q<s+it+A3#{YeOooCv%Yo-znB9QRviqpptuLIj?0e{T'
            'b(h{EAv)wFInfHRm;k-e^Px^1|J{<k>QvTlLFVp5uv)3q4@svb55e(+1{NT%n9~!{32i<Aw2RvyNEH+v0_g7u0_i^-Tq_EDP?w&p'
            'V75k<cWs9=U+JA)U7p!C%h}aE7!W<6h=U&l<rq4{u!yT42IiPYgV2<f^dLCTngV#^6C1)Va;TiZRt)>uSk5kaC>cvBJzKnTlzbrK'
            ')VaTG=UiJ*-1xbMfwe+TWzsz6W^zn-nZ&<5n;)E58O0OZ{0t61+5?MygEb#j$_L%pvVwn<N{7`G`6Aev?uj!y69MLQ>qm5n@GLaR'
            'kblbcOM|DrMQlClw0Z2s;ku~ftVeiSXwS&=4(1bXJP}Qv*30n@C_2WYld8hoSDW$t=^Zt=7e$_a%;a9G>Kdr06S|yb2ydD0)~M<t'
            '`SLup%i*+C)@RC!3{PP<wY{S|{&{(Z8sY3}F@KiaQ*FsMRe>9fZI~pCoS2Up!phiD9lN4_+IeTm@74|RE~Wk%7gN2f(kIXeC!JfY'
            '-j!>HrSDh!1n4xhFHcXOEnuUm8?7hqM>iRGfqX&QgXZ0a9=hu6pyuI!<rFO%yrt~ljvcrmyX;hz8DyLePHg}>^X*I@2ruZBE<C00'
            ')SlCwTGR6r@d71nljc~f3pOGiOKh-c&{ihx5dh_@K@%2Gu)$1KXTE+KTL3|{XF>&?VC%A2P{o2e+nzV>>79Rvjqdxl|7f(%Vi0kT'
            'L^IO*O))JFqF7b3ypdERiVH8<MMkOMNri&p2*cM!1FM;kU^PBlg|i4MnQh2jYB5t4rbt6zuDj{MQ{x_MUC7JK@>4z0@gqPXvI>%F'
            'OKNYRb<b@}?CZ5wxyaQve)jweF3dyBg{=TK(=SP{*?U&if@j=ZsS8q3RG^I&@3>rzss11Im+^iva-FCq-yzfB-Ozx&&4p??!=k>y'
            '`TW`E6c2g!IY^V0Qv|g|X4V|g-UtYulaZ|6Z{-L6%~Z6rq<^F?;i!4rih;@wFGvf!L;yZ`S5qx>q=Eopns9M2s&;F%w_K-{DshXO'
            '#Ow{=MvzNJO-tJUD5|mK@+@ig2g|MV@xjDsEPNTo)jCb}MPKydDKz9=5w0ZOH^%E#%V0b~;kFoKew^IIfWi_jB~E~wF&4qNsW;QV'
            'jvX?3ORew?d+54q(#$xid`b!g2u6JqVZy4DOI5T?cST)FIR+Dj2Z}Cuy*jdWr=I@S=~P)Yj+MZ}tx}oWc4PgHDo<-8(%jo`sb|KF'
            'Ywr9XY$`3)Ma1r%8mcxdIq8?n9%~rlSDz$Q68u*IDkj~;Yn>vE`Of+g+Gph8Ap1}Oc7%&r_}1}o1^#rNZg5BHL?)`q?hw^jqpKrT'
            'G=72Nn$io4Oj0#uJbDJyORC4j8PHVGjVl&&RuY%y#?<2`ucXW|@2N^boWLx#O`i1TAXgwe_c1FTj6~glv49)aPIZa6QD^s=b6f|='
            '$%}ZM!s``utWr>CYmD=L#8O!{uzSPov{<FmuNKdFmr?ENVY<_)YPH|L{r=+3`{>7upI=}6a`7tq+dr@#oT@bXrIr?8Go||!OH;>K'
            'JGYF>6s*nOV#Ip0Aj1Q;DJzxEUI_o3V#4n(zPosd<<9gxUcC7?aVS2fST1!125}wlsYvTUg;u?stD{Ev4l&kt2?tWbg{A{)hSp&B'
            '`Qg%D-m$txM#srEO;mAcZ{!n`^y;zoO?i3ejB#((Wc$+dxHsdyyurgdY^aK}ag^1=D0E7-*)q+_fkt9cV<HXB4~q-*N<UR-KxH$7'
            'I?Rh(ERb)2UQ&M>ka>w0yoZTrYUcSsAA2dblu3fwGka8khAuv|`?fB$yQCJTr<fIYQ6`NFj*oeYk}-CMH#L4;F}d{TeCeQYHR^E;'
            'O|?yRvTqscu5-+~({}pIoAe`iUq`N}Xx{1^Z9No<cgOuE)2y!uW-r{Hu5NKe*E8urcvQ(tkN8^cBt>;Z0n9}(Z6xeMm>oj7hG@sz'
            'p<>v+8h`W-_fVyR<>)@-5?QuvRi<4I(?;Si<-Nb{$KS*s#knfB4zAKu4Fs&UI>m~s0yY93L!yg;MH0)QN{!Z8snS(+Mp@+Jg*dm)'
            'KDDtxm2EKB*S9mqZrW?zVD5wUoelqOq++EDp_y4}B=Ck~%Edu@0n7vmCNGwjx3pK5qz=Fyl|+RwtUWc{99nwmB&Yi`3_q5=@HS|t'
            'LD&zW%|hwzZ?4YxsTQ3mh!3FKVhF|%3jX`f4CsiigE&3)+nU988>`1BGi}@+)+@i4BNG?h&_zemQ7p2j*oP}1@FuPEV5qf`VRjGm'
            '&`Q>@c(GTvVoH*F?ld)KA~j%hu&+T}w+-+w`nHDp7|#rpV4%OpGi&NKO7;Rj$AmJv1C4zeMNRwBgJo03!=5jI`yFm^H~>?Lkr;?Z'
            'czIB<tGPjk(#Xlnr`<JX$1oV^#%*Y7clQn!=yJ8A-u3f?UI1v}ddij%!t)Qx7%BHd4fV&x)|RMU_*LKTbt(xv$HwpJ_m9n=wqtzC'
            'X+G`4wi)8U4EC3KIN6x@IG$P@P_tB2a6w~$z9$$;6{jxOaVGKJ2|9l9rY%Qm9J))VpmJJa1~>+q0t1k*6L)hOE;|ZK<g&`?D<#Fp'
            'tl5b~UZ>gk%ARoIoa6Zla^h>niP3zM0i|?83r*Y|K5_T+<M!3W<4wB;8(QL<RAvbqxIDjwZH)0-Wt6u(;;<vHgg$}Wbcw2JN}t^k'
            'wEHt5cU8J;ZS9U*t*VmKW7i<Z;Du5t4&6g6@px{fLOVEhQF6P(I;*PI-q&HgLEWXLF}pU;AQDb*(LsrMZD?pMuxLzUK|L^GYfo{1'
            '9Y{(1(WsC+7N2!TwYkr)iZwX49lHRlTMc3}p>bWzKn{y4@oFkWSnpg<$B0vt$c)lw5HY_Swe~5|9j#9+KW@ODo{t=}Pr{oqNzf0o'
            'S+^_qFr)s{<6;UU%B4vHr%`xfm*Xe=Sp2E5VnhM%<?pf#2KLVh2}ygZf9ydbcfx!~Tkxs;fSPu>l-IJC=7UiG{J_*B!*;ER>C;Z('
            'W8!$j!#Ib`y}QF5&%{ZGA~6pI*B&#5ijTEf5tUX^$6_}QAESq?z))9a_L=WtrwV5WQ;rD`SRF5tc_vG;qrfNu1pTN{R5klO-;O`M'
            'n7&N)0VvdI#c?}0vwSRdXM}q%$*~5CtTI{I3U(lx<>J&6)%xSnDd_oNEUG8kG_X`XA*Qi?7(Pg6&zU5O4+ot75H2imK;d9Tz#=bW'
            'mQY|<j~^4kdhCo%fXY3yQ(MM9ma|2cma)NLYLuNluT#2IYE|cM?*mncxPG#G>35<N$CXFZ`QC)d9i^1Phrcn={d}o=lezDU6AuH_'
            'Gmq5Q0U2BOEU<eL_&21tItcXD6skV_xCHf>iFa%jy5AN(^S+OJ#<><Lw`;MQs)IK+FXdsvl?qf{`ns5EWAmy1x!ksh=a5!&&+3hj'
            '(ig&-bfZ?2o?zUh6JGN+7S#_kVR+i8G;ZfEo#$p!zljL&yqH%u47){>`;Q9s<WWX0@2rtIpDX0=@mQXd{v08hatm=sgSVdzwJ;2Z'
            '^S8b}mA@4}7-)Yy-ulL|`c$SqwzI9@K8!p0yDh@qeBnP6U;X{uest{;k41%Nmr=nxrj;I?Rz%ssMdp9*gA^6?nNg8kZtAxgIhpGu'
            'J6xDu$<z<Gr=6;0ug~nAOHq8W9wYC47w`K6`TgI~?vLklRe<*UOzkZYONwAwlrJ&Si$2DzofieaqA%BjG~hHBeMG{!0_x-JHVS<z'
            '1TSg!JhpnZ+6C)Tw|BkmscB`xAyD;)4TE(H0^r~JqKuvM1)B|YaMk0VwDVb^p>Va&kr@2-9{DlF4!<ZQ`WOz&baoyOBGbpt0KD(9'
            '*V9$0(8qny*V@*!y2FY4spj(rQG<DGCbk{)l&|*-9Jq3dD#XUCs>mAP>SD90BsvzDZ|yGHmpv)+T}k}ZliLJ-uD&$xK<`<Jo&kc}'
            'ZMxpFswXAP6%)DQ?xjapkIqjzRJ`gNqS%np0kh@&_wY;CPDj(03+ZYU&<#_cFxO}1Io2zb7;~g=mCw8j0R2ZMw)g9V{O#+MSo*7D'
            'QCTJ3SS((SJB=t`l}-gZuR`|=cMJmT-d7jjy!h$6_t7sGumAq-d+W1uf7a#Gf5OF6eNupF#LTUlvmX9FTge~O;Tr|LhA@Ahdf4b4'
            'GLxa-v(M>frus@tH2h7SLEyS;-#2&6%M)MO8J#ZOI#^Y|Q?O{bGSTSK&Wo;CWZw%=AA)giiVTC)1>ai@sPA67pAhQ$7T<zCn^JoI'
            '1oQ8%GdWe+oYPjLbNP6aaC%4&;B11!669R2hHC@yH+4+)9sm|D+Hc>f&opn+>!wsW-xKRoOCb7`1I4tjcqJ<H=^a&GoQrp>5;T<^'
            '@|l>sJe3`c;Oz2=U3~m8p46x<znWGmS2AbwWj>FNMN>EQeJBpi;%9c~%R{sDaLiYf^IZupjn<Zh6+iy;Mt$D%#hVx3{o~#1cVCM6'
            'xwkjbu~r7H4GdAB;v`e9frCqCa0242Fr6~!6#bgG-;M}XP3=k)x4E@;?ONOG2Ll!dOlYTsQJv8aOS46pPf5tS>XVzJ`HA>db1r>F'
            '7%R-`5>7f(p0m4WocA6+Rhjys`p~c*OMyFJ{yjY0QergL#>0xYDm^tjqV3Lx<CpgLz^NN@4l){j)4VDaKy3BXYx)*3xL|^HDdB?n'
            'g7m|pxtOd{gI>ei3RK|_Y{fLouo?t?Ky>zJ+>26QQ#RsOr&n2zLch&pFGg90<<c4Bsj+@=d$bpftP=hwJKV-M<B@&6ikA@sVU!`R'
            'j!xA#f$^?|{sfy9E_~>-O%Hxsbm+69_9sOj4E6L{N{)fqN?{v2cj7b{PWc`3+YUS}v5h}c#%)h)rJ<nxX6$~8ZF<+$iuau0Wl_%k'
            'LYos^Q{Nuze8kSH0px$pZrL5SrJDK}nyYlZkD~P{-H8`@C*%~jdNj39u|J_NpVODhtx9Yu7ZBUTb)12(fu%Il^^~^dQ`lltRsW53'
            '74Oq+tmOyz0(ifIKc&jX8rP^YMdH{sR*J$ZM|W148kq`~u6Ren5xRi63$lh2)){M6+=B{AVb#yq?|=Z%3$JAqxhaYFN$a}XwHf1^'
            'IL#2<rPA3IRwD<1IbUWG+|G*DX(0*ol`PVv<jUwM>4DOK6(B_vxk2FyswvKj>qernj?#>BtY_+!sf&q@Rdl~~rb0b^o$MS^Je@lG'
            '=^rnCe61Yz9jz%Qt|vvFG$Pm)c>`7-5Uo0-wV=ZiaAs0a2;W-O26sh?>9(-F&b&>1R>g_LU=P*_2Fm<~d+>?+dd^N(dx-|lkv>A7'
            '9$`+$_yCBm<K35>Qkg;Tog!`*I#cogoZ%gDAm0(`ht&&nPE<c4pxIgPZfDg?!a*aG)`$g%+ndKrM23XU%Yz?;TDq3(?||>slg7fa'
            'Xf#Cb_h_n1{2a&?9~u)@q}?vBx|eRL3KE+i6XE>9H`Hi}`fPIRa-P-gUact+FKNaDG^b6fC9+`%E60>rsZ1Y9dKQ~Xx}N_2PS}37'
            'WuiX)()PYopY^z!I1lmL9#^B@Di==qzm9e?ALHP!ufUp^dRz3QEwvTu>c+N#)!KW&mQsB}0Yo{lOu7a~rh0lluZ+mk71|Jl0!G)A'
            'fIVI+lS`XS^24FPt=M|sd|`2nwpxszIH0;Gq)3cahT=-sb)-_s<k;wl0@wl#Y+sPvMlO*tBHo&>7sTS(T+96S@+f^A8~wKBa}tkW'
            'vSS7V)hGpUYr1L079N^uFl$}2*(8c#K2g%TSCy(TbUvYjcLJ^?qVdDsZ`p96RV_1z9X2jU)8T=y`-)7O(vVyVx7+J!FYZpDzISq8'
            '+o@Ns&x+Y{r^{&4w_19^PR04DyT#5RNL{-$=ZW%`pNMa82OxtU3UthqGTi|0)a*2283j`dw5@wzZHq>w!cnlty<DuY7aM@IbxIro'
            'Gj6~Vrab^0r@m6d1_#HuEkIu7R>R!L8ep6?I7GZIi%L7le2tg+dSJb|u+?+D?=8aH1>rqX+C8kgrfrWwwpG2IdZ^-45Ga2Q`wByK'
            '%T9Mb!FsyT^rWxO&rq0{cQyUr4#g~O&S}wJ_C@zk)@oYzlf|}Hl?Bv$*4gu6>M?klc+Jf!$_v8aXdk@`R_MSDAe7T82Bme3(XIij'
            '9B&)ZoszI$@2RLqMXtr2Ul60c)~>!y&W6wL@W~#mmoiu?<$;PQZAA=FCV}UAS_Lx91jEObSkcJ)M`b6H82T<t_h&FHALYJTrp@nS'
            'QGMF9^n%ZpTjjXF%ZHO5me%W0zX!a0&>>!)ID_c~)R+Fbm=>*ceW9x)Iu|c6(|}t0%<Q++UnmI1fcv$qr(|zTwW`JM5}a1OL;OSr'
            ')OeGAkja#rPK{VVsNvA@n0IvL)YIyyvz}tK&e^5&4P(3TbElhgpy`&%HG;pVn^rzOq{eC@rx)wq7K!cAOP^mzr!RdkhWnbY`FdT^'
            'cIFH$(;|Erit};a2O&DGe}~iCqTrsLUx_D6G5%8g^5VxguiyOLKEM{Ycz>Eg2mv0K>5BF>rJRN1fwL*+Hb_sluUB_s_!fs%GF}xN'
            'fX#wtHUrXEGy+C3Rimov?C&rrOdsEj^P?TE;IKrXrr1U;LJm>j_cJTskl>I!n!@A;lyU<OgP=-R#aeP0B?cN~c&UlZv~2h$iB(=S'
            '(0LKgH4gi<(M>MIZYo2SsCe?81Y%l&b~<8Q9NY}L6|oR!SgiH_%uN$UAKu(Ta>gIH{9^PFE)gYv;wH5>=zK8S@5~zDjDjv?QBWGH'
            'lf2{F<T?*Tt6s`kBD!*7=|ns}dy6g%PRt!a=U8*>e(*j`KJ*va(vw|u3TSrjytpWLv$y=4Fx-66s{#CFwupIo2gu%EbuYx=`lrsd'
            '7&npf{{t)>gmM'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
